In [63]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix

# Import the necessary function from threadpoolctl
from threadpoolctl import threadpool_limits

# Load and preprocess the data (same as before)
data_path = "C:/Users/olufe/projects/Journal/dataset/HomeA_unsupervised_sim_combined_shuffled.csv"
data = pd.read_csv(data_path)
X = data.drop(columns=["Label"])
y = data["Label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [64]:
# Build the Deep Autoencoder

def build_autoencoder(input_dim):
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(64, activation='relu')(input_layer)
    encoded = Dense(32, activation='relu')(encoded)
    encoded = Dense(16, activation='relu')(encoded)
    decoded = Dense(32, activation='relu')(encoded)
    decoded = Dense(64, activation='relu')(decoded)
    decoded = Dense(input_dim, activation='linear')(decoded)

    autoencoder = Model(input_layer, decoded)
    autoencoder.compile(optimizer='adam', loss='mean_squared_error')
    return autoencoder

# Assuming the input dimension is the number of features in the dataset
input_dim = X_train_scaled.shape[1]
autoencoder = build_autoencoder(input_dim)

In [65]:
# Train the autoencoder (same as before)
autoencoder = build_autoencoder(input_dim)
early_stopping = EarlyStopping(patience=3, restore_best_weights=True)
autoencoder.fit(X_train_scaled, X_train_scaled, epochs=50, batch_size=64, validation_split=0.1, callbacks=[early_stopping])

Epoch 1/50
398/398 [==============================] - 9s 9ms/step - loss: 0.7558 - val_loss: 0.6337
Epoch 2/50
398/398 [==============================] - 3s 9ms/step - loss: 0.5855 - val_loss: 0.5485
Epoch 3/50
398/398 [==============================] - 3s 8ms/step - loss: 0.5210 - val_loss: 0.5065
Epoch 4/50
398/398 [==============================] - 3s 9ms/step - loss: 0.4866 - val_loss: 0.4767
Epoch 5/50
398/398 [==============================] - 3s 8ms/step - loss: 0.4669 - val_loss: 0.4620
Epoch 6/50
398/398 [==============================] - 3s 9ms/step - loss: 0.4539 - val_loss: 0.4524
Epoch 7/50
398/398 [==============================] - 3s 8ms/step - loss: 0.4437 - val_loss: 0.4448
Epoch 8/50
398/398 [==============================] - 3s 8ms/step - loss: 0.4368 - val_loss: 0.4418
Epoch 9/50
398/398 [==============================] - 4s 9ms/step - loss: 0.4308 - val_loss: 0.4329
Epoch 10/50
398/398 [==============================] - 4s 9ms/step - loss: 0.4257 - val_loss: 0.4345

In [66]:
# Define the number of clusters for k-means
num_clusters = 5

# Initialize k-means with data points and get cluster memberships
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
X_train_clusters = kmeans.fit_predict(X_train_scaled)

# Create separate autoencoders for each cluster
cluster_autoencoders = []
for cluster_id in range(num_clusters):
    cluster_samples = X_train_scaled[X_train_clusters == cluster_id]
    cluster_autoencoder = build_autoencoder(input_dim)
    cluster_autoencoder.fit(cluster_samples, cluster_samples, epochs=50, batch_size=64, validation_split=0.1, callbacks=[early_stopping])
    cluster_autoencoders.append(cluster_autoencoder)

Epoch 1/50
203/203 [==============================] - 3s 8ms/step - loss: 0.2779 - val_loss: 0.2384
Epoch 2/50
203/203 [==============================] - 2s 8ms/step - loss: 0.2150 - val_loss: 0.2027
Epoch 3/50
203/203 [==============================] - 2s 8ms/step - loss: 0.1885 - val_loss: 0.1812
Epoch 4/50
203/203 [==============================] - 2s 8ms/step - loss: 0.1735 - val_loss: 0.1706
Epoch 5/50
203/203 [==============================] - 2s 8ms/step - loss: 0.1650 - val_loss: 0.1654
Epoch 6/50
203/203 [==============================] - 2s 8ms/step - loss: 0.1593 - val_loss: 0.1604
Epoch 7/50
203/203 [==============================] - 2s 8ms/step - loss: 0.1550 - val_loss: 0.1567
Epoch 8/50
203/203 [==============================] - 2s 8ms/step - loss: 0.1517 - val_loss: 0.1528
Epoch 9/50
203/203 [==============================] - 2s 8ms/step - loss: 0.1488 - val_loss: 0.1508
Epoch 10/50
203/203 [==============================] - 2s 8ms/step - loss: 0.1465 - val_loss: 0.1488

Epoch 33/50
52/52 [==============================] - 0s 8ms/step - loss: 0.7656 - val_loss: 0.8290
Epoch 34/50
52/52 [==============================] - 0s 8ms/step - loss: 0.7605 - val_loss: 0.8226
Epoch 35/50
52/52 [==============================] - 0s 8ms/step - loss: 0.7566 - val_loss: 0.8186
Epoch 36/50
52/52 [==============================] - 0s 9ms/step - loss: 0.7521 - val_loss: 0.8135
Epoch 37/50
52/52 [==============================] - 0s 8ms/step - loss: 0.7488 - val_loss: 0.8096
Epoch 38/50
52/52 [==============================] - 0s 8ms/step - loss: 0.7447 - val_loss: 0.8120
Epoch 39/50
52/52 [==============================] - 0s 7ms/step - loss: 0.7428 - val_loss: 0.8086
Epoch 40/50
52/52 [==============================] - 0s 8ms/step - loss: 0.7385 - val_loss: 0.8024
Epoch 41/50
52/52 [==============================] - 0s 9ms/step - loss: 0.7357 - val_loss: 0.8027
Epoch 42/50
52/52 [==============================] - 0s 8ms/step - loss: 0.7323 - val_loss: 0.8023
Epoch 43/5

Epoch 16/50
23/23 [==============================] - 0s 7ms/step - loss: 0.7284 - val_loss: 0.7500
Epoch 17/50
23/23 [==============================] - 0s 7ms/step - loss: 0.7126 - val_loss: 0.7354
Epoch 18/50
23/23 [==============================] - 0s 7ms/step - loss: 0.6933 - val_loss: 0.7209
Epoch 19/50
23/23 [==============================] - 0s 8ms/step - loss: 0.6798 - val_loss: 0.7068
Epoch 20/50
23/23 [==============================] - 0s 7ms/step - loss: 0.6664 - val_loss: 0.6980
Epoch 21/50
23/23 [==============================] - 0s 8ms/step - loss: 0.6537 - val_loss: 0.6852
Epoch 22/50
23/23 [==============================] - 0s 7ms/step - loss: 0.6422 - val_loss: 0.6775
Epoch 23/50
23/23 [==============================] - 0s 7ms/step - loss: 0.6291 - val_loss: 0.6632
Epoch 24/50
23/23 [==============================] - 0s 8ms/step - loss: 0.6175 - val_loss: 0.6539
Epoch 25/50
23/23 [==============================] - 0s 7ms/step - loss: 0.6088 - val_loss: 0.6451
Epoch 26/5

Epoch 49/50
61/61 [==============================] - 0s 6ms/step - loss: 0.4018 - val_loss: 0.4231
Epoch 50/50
61/61 [==============================] - 0s 7ms/step - loss: 0.4005 - val_loss: 0.4239


In [67]:
# Perform self-training iterations
num_self_training_iterations = 3
for iteration in range(num_self_training_iterations):
    # Obtain reconstruction errors for all samples using the ensemble
    ensemble_reconstruction_errors = np.zeros(X_train_scaled.shape[0])
    for cluster_id, cluster_autoencoder in enumerate(cluster_autoencoders):
        cluster_samples = X_train_scaled[X_train_clusters == cluster_id]
        reconstruction_errors = np.mean(np.square(cluster_samples - cluster_autoencoder.predict(cluster_samples)), axis=1)
        ensemble_reconstruction_errors[X_train_clusters == cluster_id] = reconstruction_errors

    # Identify unlabeled anomalies and add them to the training set
    threshold = np.percentile(ensemble_reconstruction_errors, 95)
    unlabeled_anomalies_indices = np.where(ensemble_reconstruction_errors > threshold)[0]
    if len(unlabeled_anomalies_indices) == 0:
        break
    X_train_scaled = np.concatenate((X_train_scaled, X_train_scaled[unlabeled_anomalies_indices]), axis=0)

    # Update k-means clustering with the updated training set
    X_train_clusters = kmeans.fit_predict(X_train_scaled)

    # Update cluster autoencoders with the updated training set
    for cluster_id in range(num_clusters):
        cluster_samples = X_train_scaled[X_train_clusters == cluster_id]
        cluster_autoencoder = build_autoencoder(input_dim)
        cluster_autoencoder.fit(cluster_samples, cluster_samples, epochs=50, batch_size=64, validation_split=0.1, callbacks=[early_stopping])
        cluster_autoencoders[cluster_id] = cluster_autoencoder

135/135 [==============================] - 0s 2ms/step
Epoch 1/50
208/208 [==============================] - 2s 6ms/step - loss: 0.2914 - val_loss: 0.2720
Epoch 2/50
208/208 [==============================] - 1s 6ms/step - loss: 0.2271 - val_loss: 0.2339
Epoch 3/50
208/208 [==============================] - 1s 5ms/step - loss: 0.1984 - val_loss: 0.2098
Epoch 4/50
208/208 [==============================] - 1s 6ms/step - loss: 0.1816 - val_loss: 0.1967
Epoch 5/50
208/208 [==============================] - 1s 6ms/step - loss: 0.1727 - val_loss: 0.1895
Epoch 6/50
208/208 [==============================] - 1s 5ms/step - loss: 0.1665 - val_loss: 0.1815
Epoch 7/50
208/208 [==============================] - 1s 5ms/step - loss: 0.1619 - val_loss: 0.1769
Epoch 8/50
208/208 [==============================] - 1s 6ms/step - loss: 0.1585 - val_loss: 0.1742
Epoch 9/50
208/208 [==============================] - 1s 5ms/step - loss: 0.1555 - val_loss: 0.1708
Epoch 10/50
208/208 [========================

63/63 [==============================] - 0s 6ms/step - loss: 0.7201 - val_loss: 0.9761
Epoch 47/50
63/63 [==============================] - 0s 6ms/step - loss: 0.7152 - val_loss: 0.9713
Epoch 48/50
63/63 [==============================] - 0s 5ms/step - loss: 0.7164 - val_loss: 0.9742
Epoch 49/50
63/63 [==============================] - 0s 6ms/step - loss: 0.7136 - val_loss: 0.9730
Epoch 50/50
63/63 [==============================] - 0s 6ms/step - loss: 0.7131 - val_loss: 0.9636
Epoch 1/50
65/65 [==============================] - 2s 9ms/step - loss: 1.1339 - val_loss: 1.2132
Epoch 2/50
65/65 [==============================] - 0s 6ms/step - loss: 0.8373 - val_loss: 1.0001
Epoch 3/50
65/65 [==============================] - 0s 6ms/step - loss: 0.7220 - val_loss: 0.9041
Epoch 4/50
65/65 [==============================] - 0s 6ms/step - loss: 0.6626 - val_loss: 0.8263
Epoch 5/50
65/65 [==============================] - 0s 5ms/step - loss: 0.6171 - val_loss: 0.7850
Epoch 6/50
65/65 [=========

21/21 [==============================] - 0s 6ms/step - loss: 0.6703 - val_loss: 1.0108
Epoch 30/50
21/21 [==============================] - 0s 6ms/step - loss: 0.6532 - val_loss: 0.9969
Epoch 31/50
21/21 [==============================] - 0s 7ms/step - loss: 0.6474 - val_loss: 0.9863
Epoch 32/50
21/21 [==============================] - 0s 6ms/step - loss: 0.6500 - val_loss: 0.9818
Epoch 33/50
21/21 [==============================] - 0s 6ms/step - loss: 0.6312 - val_loss: 0.9649
Epoch 34/50
21/21 [==============================] - 0s 6ms/step - loss: 0.6234 - val_loss: 0.9579
Epoch 35/50
21/21 [==============================] - 0s 6ms/step - loss: 0.6176 - val_loss: 0.9527
Epoch 36/50
21/21 [==============================] - 0s 7ms/step - loss: 0.6223 - val_loss: 0.9484
Epoch 37/50
21/21 [==============================] - 0s 6ms/step - loss: 0.6134 - val_loss: 0.9390
Epoch 38/50
21/21 [==============================] - 0s 6ms/step - loss: 0.6046 - val_loss: 0.9328
Epoch 39/50
21/21 [===

229/229 [==============================] - 1s 6ms/step - loss: 0.1737 - val_loss: 0.3074
Epoch 11/50
229/229 [==============================] - 1s 6ms/step - loss: 0.1715 - val_loss: 0.3048
Epoch 12/50
229/229 [==============================] - 1s 6ms/step - loss: 0.1694 - val_loss: 0.3040
Epoch 13/50
229/229 [==============================] - 1s 6ms/step - loss: 0.1677 - val_loss: 0.3017
Epoch 14/50
229/229 [==============================] - 1s 6ms/step - loss: 0.1662 - val_loss: 0.3008
Epoch 15/50
229/229 [==============================] - 1s 6ms/step - loss: 0.1649 - val_loss: 0.2995
Epoch 16/50
229/229 [==============================] - 1s 6ms/step - loss: 0.1636 - val_loss: 0.2985
Epoch 17/50
229/229 [==============================] - 1s 6ms/step - loss: 0.1624 - val_loss: 0.2973
Epoch 18/50
229/229 [==============================] - 1s 6ms/step - loss: 0.1616 - val_loss: 0.2970
Epoch 19/50
229/229 [==============================] - 1s 6ms/step - loss: 0.1608 - val_loss: 0.2968
Ep

38/38 [==============================] - 0s 6ms/step - loss: 0.7792 - val_loss: 1.2159
Epoch 16/50
38/38 [==============================] - 0s 6ms/step - loss: 0.7618 - val_loss: 1.1802
Epoch 17/50
38/38 [==============================] - 0s 7ms/step - loss: 0.7471 - val_loss: 1.1677
Epoch 18/50
38/38 [==============================] - 0s 6ms/step - loss: 0.7341 - val_loss: 1.1396
Epoch 19/50
38/38 [==============================] - 0s 6ms/step - loss: 0.7214 - val_loss: 1.1177
Epoch 20/50
38/38 [==============================] - 0s 6ms/step - loss: 0.7120 - val_loss: 1.1003
Epoch 21/50
38/38 [==============================] - 0s 5ms/step - loss: 0.7002 - val_loss: 1.0855
Epoch 22/50
38/38 [==============================] - 0s 6ms/step - loss: 0.6894 - val_loss: 1.0722
Epoch 23/50
38/38 [==============================] - 0s 5ms/step - loss: 0.6796 - val_loss: 1.0594
Epoch 24/50
38/38 [==============================] - 0s 6ms/step - loss: 0.6725 - val_loss: 1.0450
Epoch 25/50
38/38 [===

5/5 [==============================] - 0s 14ms/step - loss: 0.7192 - val_loss: 0.7908
Epoch 49/50
5/5 [==============================] - 0s 12ms/step - loss: 0.7094 - val_loss: 0.7848
Epoch 50/50
5/5 [==============================] - 0s 11ms/step - loss: 0.7025 - val_loss: 0.7752
Epoch 1/50
69/69 [==============================] - 1s 7ms/step - loss: 1.8827 - val_loss: 2.1662
Epoch 2/50
69/69 [==============================] - 0s 5ms/step - loss: 1.5025 - val_loss: 1.8853
Epoch 3/50
69/69 [==============================] - 0s 5ms/step - loss: 1.3249 - val_loss: 1.7144
Epoch 4/50
69/69 [==============================] - 0s 5ms/step - loss: 1.2142 - val_loss: 1.5807
Epoch 5/50
69/69 [==============================] - 0s 5ms/step - loss: 1.1345 - val_loss: 1.4725
Epoch 6/50
69/69 [==============================] - 0s 5ms/step - loss: 1.0772 - val_loss: 1.3871
Epoch 7/50
69/69 [==============================] - 0s 5ms/step - loss: 1.0311 - val_loss: 1.3195
Epoch 8/50
69/69 [==============

53/53 [==============================] - 0s 5ms/step - loss: 0.5981 - val_loss: 1.0424
Epoch 29/50
53/53 [==============================] - 0s 5ms/step - loss: 0.5921 - val_loss: 1.0320
Epoch 30/50
53/53 [==============================] - 0s 5ms/step - loss: 0.5885 - val_loss: 1.0344
Epoch 31/50
53/53 [==============================] - 0s 5ms/step - loss: 0.5860 - val_loss: 1.0288
Epoch 32/50
53/53 [==============================] - 0s 5ms/step - loss: 0.5814 - val_loss: 1.0225
Epoch 33/50
53/53 [==============================] - 0s 5ms/step - loss: 0.5797 - val_loss: 1.0142
Epoch 34/50
53/53 [==============================] - 0s 5ms/step - loss: 0.5763 - val_loss: 1.0053
Epoch 35/50
53/53 [==============================] - 0s 5ms/step - loss: 0.5727 - val_loss: 1.0004
Epoch 36/50
53/53 [==============================] - 0s 5ms/step - loss: 0.5715 - val_loss: 0.9974
Epoch 37/50
53/53 [==============================] - 0s 5ms/step - loss: 0.5691 - val_loss: 0.9979
Epoch 38/50
53/53 [===

73/73 [==============================] - 0s 5ms/step - loss: 0.9172 - val_loss: 1.1626
Epoch 12/50
73/73 [==============================] - 0s 5ms/step - loss: 0.8980 - val_loss: 1.1304
Epoch 13/50
73/73 [==============================] - 0s 5ms/step - loss: 0.8793 - val_loss: 1.0941
Epoch 14/50
73/73 [==============================] - 0s 5ms/step - loss: 0.8638 - val_loss: 1.0779
Epoch 15/50
73/73 [==============================] - 0s 5ms/step - loss: 0.8515 - val_loss: 1.0614
Epoch 16/50
73/73 [==============================] - 0s 5ms/step - loss: 0.8387 - val_loss: 1.0452
Epoch 17/50
73/73 [==============================] - 0s 5ms/step - loss: 0.8297 - val_loss: 1.0328
Epoch 18/50
73/73 [==============================] - 0s 5ms/step - loss: 0.8200 - val_loss: 1.0200
Epoch 19/50
73/73 [==============================] - 0s 5ms/step - loss: 0.8126 - val_loss: 1.0071
Epoch 20/50
73/73 [==============================] - 0s 5ms/step - loss: 0.8063 - val_loss: 0.9938
Epoch 21/50
73/73 [===

6/6 [==============================] - 0s 14ms/step - loss: 1.8659 - val_loss: 1.8994
Epoch 8/50
6/6 [==============================] - 0s 11ms/step - loss: 1.7155 - val_loss: 1.8186
Epoch 9/50
6/6 [==============================] - 0s 12ms/step - loss: 1.5954 - val_loss: 1.7515
Epoch 10/50
6/6 [==============================] - 0s 10ms/step - loss: 1.4880 - val_loss: 1.7122
Epoch 11/50
6/6 [==============================] - 0s 10ms/step - loss: 1.3921 - val_loss: 1.6552
Epoch 12/50
6/6 [==============================] - 0s 10ms/step - loss: 1.2988 - val_loss: 1.5997
Epoch 13/50
6/6 [==============================] - 0s 10ms/step - loss: 1.2265 - val_loss: 1.5403
Epoch 14/50
6/6 [==============================] - 0s 10ms/step - loss: 1.1694 - val_loss: 1.4671
Epoch 15/50
6/6 [==============================] - 0s 10ms/step - loss: 1.1278 - val_loss: 1.4068
Epoch 16/50
6/6 [==============================] - 0s 11ms/step - loss: 1.0907 - val_loss: 1.3672
Epoch 17/50
6/6 [=================

In [68]:
# Now let's evaluate the final model on the test set
y_test_pred = []
# Initialize k-means for test data
kmeans_test = KMeans(n_clusters=num_clusters, random_state=42)
# Fit k-means to test data and get cluster memberships
X_test_clusters = kmeans_test.fit_predict(X_test_scaled)

for cluster_id in range(num_clusters):
    # Get the indices of the cluster samples in the test set
    cluster_indices = np.where(X_test_clusters == cluster_id)[0]

    # Check if there are cluster samples in the test set
    if len(cluster_indices) == 0:
        continue  # Skip this cluster if there are no samples in the test set

    cluster_samples = X_test_scaled[cluster_indices]
    cluster_autoencoder = cluster_autoencoders[cluster_id]
    reconstruction_errors = np.mean(np.square(cluster_samples - cluster_autoencoder.predict(cluster_samples)), axis=1)
    y_test_pred.extend([1 if err > threshold else 0 for err in reconstruction_errors])

35/35 [==============================] - 0s 1ms/step


In [69]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix

# Convert y_test_pred to numpy array
y_test_pred = np.array(y_test_pred)

# Calculate performance metrics
accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
tnr = tn / (tn + fp)
fpr = fp / (tn + fp)
fnr = fn / (fn + tp)
f1 = f1_score(y_test, y_test_pred)
auc = roc_auc_score(y_test, y_test_pred)

# Print the performance metrics
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("TNR:", tnr)
print("FPR:", fpr)
print("FNR:", fnr)
print("F1-score:", f1)
print("AUC:", auc)

Accuracy: 0.611944523068214
Precision: 0.005107624954396206
Recall: 0.4827586206896552
TNR: 0.6124769077731989
FPR: 0.3875230922268012
FNR: 0.5172413793103449
F1-score: 0.010108303249097474
AUC: 0.547617764231427


In [58]:
#Evaluating Semi-Supervised Ensemble model

In [59]:
# Now let's evaluate the final model on the test set
y_test_pred_semi_supervised = []
# Initialize k-means for test data
kmeans_test = KMeans(n_clusters=num_clusters, random_state=42)
# Fit k-means to test data and get cluster memberships
X_test_clusters = kmeans_test.fit_predict(X_test_scaled)

for cluster_id in range(num_clusters):
    # Get the indices of the cluster samples in the test set
    cluster_indices = np.where(X_test_clusters == cluster_id)[0]

    # Check if there are cluster samples in the test set
    if len(cluster_indices) == 0:
        continue  # Skip this cluster if there are no samples in the test set

    cluster_samples = X_test_scaled[cluster_indices]
    cluster_autoencoder = cluster_autoencoders[cluster_id]
    reconstruction_errors = np.mean(np.square(cluster_samples - cluster_autoencoder.predict(cluster_samples)), axis=1)
    y_test_pred_semi_supervised.extend([1 if err > threshold else 0 for err in reconstruction_errors])

35/35 [==============================] - 0s 2ms/step


In [60]:
# Convert y_test_pred to numpy array
y_test_pred_semi_supervised = np.array(y_test_pred_semi_supervised)

# Calculate performance metrics
accuracy_semi_supervised = accuracy_score(y_test, y_test_pred_semi_supervised)
precision_semi_supervised = precision_score(y_test, y_test_pred_semi_supervised)
recall_semi_supervised = recall_score(y_test, y_test_pred_semi_supervised)
conf_matrix_semi_supervised = confusion_matrix(y_test, y_test_pred_semi_supervised)
tn_semi_supervised, fp_semi_supervised, fn_semi_supervised, tp_semi_supervised = conf_matrix_semi_supervised.ravel()
tnr_semi_supervised = tn_semi_supervised / (tn_semi_supervised + fp_semi_supervised)
fpr_semi_supervised = fp_semi_supervised / (tn_semi_supervised + fp_semi_supervised)
fnr_semi_supervised = fn_semi_supervised / (fn_semi_supervised + tp_semi_supervised)
f1_semi_supervised = 2 * (precision_semi_supervised * recall_semi_supervised) / (precision_semi_supervised + recall_semi_supervised)
auc_semi_supervised = roc_auc_score(y_test, y_test_pred_semi_supervised)

# Print the performance metrics for Semi-Supervised Ensemble method
print("Semi-Supervised Ensemble Method")
print("Accuracy:", accuracy_semi_supervised)
print("Precision:", precision_semi_supervised)
print("Recall:", recall_semi_supervised)
print("TNR:", tnr_semi_supervised)
print("FPR:", fpr_semi_supervised)
print("FNR:", fnr_semi_supervised)
print("F1-score:", f1_semi_supervised)
print("AUC:", auc_semi_supervised)

Semi-Supervised Ensemble Method
Accuracy: 0.5795358052646477
Precision: 0.005379959650302623
Recall: 0.5517241379310345
TNR: 0.5796504192127327
FPR: 0.4203495807872673
FNR: 0.4482758620689655
F1-score: 0.010656010656010656
AUC: 0.5656872785718836


In [61]:
#Now let's implement the Self-Training method:

In [62]:
# Initialize k-means for self-training
kmeans_self_training = KMeans(n_clusters=num_clusters, random_state=42)
# Fit k-means to the entire training data and get cluster memberships
X_clusters_self_training = kmeans_self_training.fit_predict(X_train_scaled)

# Perform self-training iterations (same as before)
# ...

# Now let's evaluate the final model on the test set
y_test_pred_self_training = []
# Initialize k-means for test data
kmeans_test = KMeans(n_clusters=num_clusters, random_state=42)
# Fit k-means to test data and get cluster memberships
X_test_clusters = kmeans_test.fit_predict(X_test_scaled)

for cluster_id in range(num_clusters):
    # Get the indices of the cluster samples in the test set
    cluster_indices = np.where(X_test_clusters == cluster_id)[0]

    # Check if there are cluster samples in the test set
    if len(cluster_indices) == 0:
        continue  # Skip this cluster if there are no samples in the test set

    cluster_samples = X_test_scaled[cluster_indices]
    cluster_autoencoder = cluster_autoencoders[cluster_id]
    reconstruction_errors = np.mean(np.square(cluster_samples - cluster_autoencoder.predict(cluster_samples)), axis=1)
    y_test_pred_self_training.extend([1 if err > threshold else 0 for err in reconstruction_errors])

# Convert y_test_pred to numpy array
y_test_pred_self_training = np.array(y_test_pred_self_training)

# Calculate performance metrics
accuracy_self_training = accuracy_score(y_test, y_test_pred_self_training)
precision_self_training = precision_score(y_test, y_test_pred_self_training)
recall_self_training = recall_score(y_test, y_test_pred_self_training)
conf_matrix_self_training = confusion_matrix(y_test, y_test_pred_self_training)
tn_self_training, fp_self_training, fn_self_training, tp_self_training = conf_matrix_self_training.ravel()
tnr_self_training = tn_self_training / (tn_self_training + fp_self_training)
fpr_self_training = fp_self_training / (tn_self_training + fp_self_training)
fnr_self_training = fn_self_training / (fn_self_training + tp_self_training)
f1_self_training = 2 * (precision_self_training * recall_self_training) / (precision_self_training + recall_self_training)
auc_self_training = roc_auc_score(y_test, y_test_pred_self_training)

# Print the performance metrics for Self-Training method
print("Self-Training Method")
print("Accuracy:", accuracy_self_training)
print("Precision:", precision_self_training)
print("Recall:", recall_self_training)
print("TNR:", tnr_self_training)
print("FPR:", fpr_self_training)
print("FNR:", fnr_self_training)
print("F1-score:", f1_self_training)
print("AUC:", auc_self_training)


35/35 [==============================] - 0s 3ms/step
Self-Training Method
Accuracy: 0.5795358052646477
Precision: 0.005379959650302623
Recall: 0.5517241379310345
TNR: 0.5796504192127327
FPR: 0.4203495807872673
FNR: 0.4482758620689655
F1-score: 0.010656010656010656
AUC: 0.5656872785718836
